In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/gibberish-parameters/best_model_params.pt
/kaggle/input/tinystories-narrative-classification/validation.csv
/kaggle/input/tinystories-narrative-classification/train.csv


Step 1: Import the Dataset 

TinyStories is a synthetic dataset of short stories that only contain words that a typical 3 to 4-year-olds usually understand, generated by GPT-3.5 and GPT-4. We can get it from HuggingFace.

In [5]:
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("/kaggle/input/tinystories-narrative-classification")

In [19]:
import json
import hashlib
import random
import re
from typing import Dict, List, Set, Tuple
from collections import defaultdict

class GibberishLanguageMapper:
    def __init__(self, seed: int = 42):
        """
        Initialize the mapper with a seed for consistent gibberish generation
        """
        self.seed = seed
        random.seed(seed)
        
        # Core gibberish phonemes and patterns
        self.consonants = ['b', 'c', 'd', 'f', 'g', 'h', 'j', 'k', 'l', 'm', 
                          'n', 'p', 'q', 'r', 's', 't', 'v', 'w', 'x', 'z']
        self.vowels = ['a', 'e', 'i', 'o', 'u', 'y']
        self.syllable_patterns = ['CV', 'CVC', 'VC', 'CCV', 'VCC', 'CVCV']
        
        # Storage for mappings
        self.english_to_gibberish = {}
        self.gibberish_to_english = {}
        self.word_categories = defaultdict(list)
        
    def _hash_word(self, word: str) -> int:
        """Create a consistent hash for a word to ensure same gibberish each time"""
        return int(hashlib.md5(f"{word}{self.seed}".encode()).hexdigest()[:8], 16)
    
    def _generate_syllable(self, pattern: str, word_hash: int) -> str:
        """Generate a syllable based on pattern and word hash"""
        # Use word hash to select characters consistently
        syllable = ""
        char_index = word_hash
        
        for char_type in pattern:
            if char_type == 'C':
                consonant_idx = char_index % len(self.consonants)
                syllable += self.consonants[consonant_idx]
                char_index = char_index // len(self.consonants)
            elif char_type == 'V':
                vowel_idx = char_index % len(self.vowels)
                syllable += self.vowels[vowel_idx]
                char_index = char_index // len(self.vowels)
        
        return syllable
    
    def _generate_gibberish_word(self, english_word: str) -> str:
        """Generate a consistent gibberish word for an English word"""
        word_hash = self._hash_word(english_word.lower())
        
        # Determine number of syllables based on original word length
        if len(english_word) <= 3:
            syllable_count = 1
        elif len(english_word) <= 6:
            syllable_count = 2
        else:
            syllable_count = min(3, (len(english_word) + 2) // 3)
        
        gibberish_word = ""
        
        for i in range(syllable_count):
            # Use different hash components for each syllable
            syllable_hash = (word_hash + i * 1000) % (2**31)
            pattern_idx = syllable_hash % len(self.syllable_patterns)
            pattern = self.syllable_patterns[pattern_idx]
            
            syllable = self._generate_syllable(pattern, syllable_hash)
            gibberish_word += syllable
        
        return gibberish_word
    
    def create_word_mapping(self, english_words: List[str]) -> Dict[str, str]:
        """Create consistent English to gibberish mappings"""
        mapping = {}
        
        for word in english_words:
            clean_word = word.lower().strip('.,!?";')
            if clean_word and clean_word not in mapping:
                gibberish = self._generate_gibberish_word(clean_word)
                mapping[clean_word] = gibberish
                self.gibberish_to_english[gibberish] = clean_word
        
        self.english_to_gibberish.update(mapping)
        return mapping
    
    def translate_text(self, text: str, direction: str = 'to_gibberish') -> str:
        """Translate text between English and gibberish"""
        if direction == 'to_gibberish':
            return self._translate_to_gibberish(text)
        elif direction == 'to_english':
            return self._translate_to_english(text)
        else:
            raise ValueError("Direction must be 'to_gibberish' or 'to_english'")
    
    def debug_translate(self, text: str) -> str:
        """Debug version of translate_text that shows what's happening"""
        print(f"Debugging translation of: '{text}'")
        result = ""
        i = 0
        words_processed = []
        
        while i < len(text):
            # Check if we're at the start of a word
            if text[i].isalpha():
                # Extract the complete word
                word_start = i
                while i < len(text) and text[i].isalpha():
                    i += 1
                word = text[word_start:i]
                
                clean_word = word.lower()
                
                # Generate gibberish if not already mapped
                if clean_word not in self.english_to_gibberish:
                    self.create_word_mapping([clean_word])
                
                gibberish_word = self.english_to_gibberish[clean_word]
                words_processed.append(f"{word} → {gibberish_word}")
                
                # Preserve capitalization pattern
                if word.isupper():
                    gibberish_word = gibberish_word.upper()
                elif word.istitle():
                    gibberish_word = gibberish_word.capitalize()
                
                result += gibberish_word
            else:
                # Not a letter, so preserve as-is (spaces, punctuation, etc.)
                result += text[i]
                i += 1
        
        print(f"Words processed: {', '.join(words_processed)}")
        print(f"Result: '{result}'")
        return result
    
    def _translate_to_gibberish(self, text: str) -> str:
        """Translate English text to gibberish"""
        result = ""
        i = 0
        
        while i < len(text):
            # Check if we're at the start of a word
            if text[i].isalpha():
                # Extract the complete word
                word_start = i
                while i < len(text) and text[i].isalpha():
                    i += 1
                word = text[word_start:i]
                
                clean_word = word.lower()
                
                # Generate gibberish if not already mapped
                if clean_word not in self.english_to_gibberish:
                    self.create_word_mapping([clean_word])
                
                gibberish_word = self.english_to_gibberish[clean_word]
                
                # Preserve capitalization pattern
                if word.isupper():
                    gibberish_word = gibberish_word.upper()
                elif word.istitle():
                    gibberish_word = gibberish_word.capitalize()
                
                result += gibberish_word
            else:
                # Not a letter, so preserve as-is (spaces, punctuation, etc.)
                result += text[i]
                i += 1
        
        return result
    
    def _translate_to_english(self, text: str) -> str:
        """Translate gibberish text back to English"""
        result = ""
        i = 0
        
        while i < len(text):
            # Check if we're at the start of a word
            if text[i].isalpha():
                # Extract the complete word
                word_start = i
                while i < len(text) and text[i].isalpha():
                    i += 1
                word = text[word_start:i]
                
                clean_word = word.lower()
                
                if clean_word in self.gibberish_to_english:
                    english_word = self.gibberish_to_english[clean_word]
                    
                    # Preserve capitalization pattern
                    if word.isupper():
                        english_word = english_word.upper()
                    elif word.istitle():
                        english_word = english_word.capitalize()
                    
                    result += english_word
                else:
                    result += word  # Unknown gibberish word, keep as-is
            else:
                # Not a letter, so preserve as-is (spaces, punctuation, etc.)
                result += text[i]
                i += 1
        
        return result
    
    def create_training_dataset(self, stories: List[str], output_file: str = 'gibberish_dataset.json'):
        """Create parallel English-Gibberish dataset for training"""
        dataset = {
            'english_stories': [],
            'gibberish_stories': [],
            'word_mappings': {},
            'reverse_mappings': {}
        }
        
        for story in stories:
            # Extract unique words from all stories first
            all_words = set()
            for s in stories:
                words = re.findall(r'\b\w+\b', s.lower())
                all_words.update(words)
        
        # Create mappings for all unique words
        self.create_word_mapping(list(all_words))
        
        # Translate each story
        for story in stories:
            gibberish_story = self.translate_text(story, 'to_gibberish')
            dataset['english_stories'].append(story)
            dataset['gibberish_stories'].append(gibberish_story)
        
        # Store mappings
        dataset['word_mappings'] = self.english_to_gibberish.copy()
        dataset['reverse_mappings'] = self.gibberish_to_english.copy()
        
        # Save to file
        with open(output_file, 'w') as f:
            json.dump(dataset, f, indent=2)
        
        return dataset
    
    def create_semantic_variants(self, base_mappings: Dict[str, str]) -> Dict[str, List[str]]:
        """Create multiple gibberish variants for the same English word to test semantic understanding"""
        variants = {}
        
        for english_word, gibberish_word in base_mappings.items():
            # Create 2-3 variants by slightly modifying the base gibberish
            word_variants = [gibberish_word]  # Include original
            
            # Variant 1: Change one vowel
            variant1 = list(gibberish_word)
            for i, char in enumerate(variant1):
                if char in self.vowels:
                    # Replace with different vowel
                    new_vowels = [v for v in self.vowels if v != char]
                    if new_vowels:
                        variant1[i] = random.choice(new_vowels)
                    break
            word_variants.append(''.join(variant1))
            
            # Variant 2: Add/remove a consonant
            if len(gibberish_word) > 2:
                variant2 = gibberish_word[:-1]  # Remove last character
            else:
                variant2 = gibberish_word + random.choice(self.consonants)  # Add consonant
            word_variants.append(variant2)
            
            variants[english_word] = word_variants
        
        return variants
    
    def export_mappings(self, filename: str = 'language_mappings.json'):
        """Export all mappings to a file"""
        export_data = {
            'english_to_gibberish': self.english_to_gibberish,
            'gibberish_to_english': self.gibberish_to_english,
            'generation_seed': self.seed,
            'phoneme_inventory': {
                'consonants': self.consonants,
                'vowels': self.vowels,
                'syllable_patterns': self.syllable_patterns
            }
        }
        
        with open(filename, 'w') as f:
            json.dump(export_data, f, indent=2)
        
        print(f"Exported {len(self.english_to_gibberish)} word mappings to {filename}")
    
    def load_mappings(self, filename: str):
        """Load previously created mappings"""
        with open(filename, 'r') as f:
            data = json.load(f)
        
        self.english_to_gibberish = data['english_to_gibberish']
        self.gibberish_to_english = data['gibberish_to_english']
        self.seed = data['generation_seed']
        
        if 'phoneme_inventory' in data:
            phonemes = data['phoneme_inventory']
            self.consonants = phonemes['consonants']
            self.vowels = phonemes['vowels']
            self.syllable_patterns = phonemes['syllable_patterns']


def demonstrate_usage():
    """Demonstrate the gibberish language mapper"""
    # Sample tiny stories
    sample_stories = [
        "The little boy ran to the park. He was very happy and played with his red ball.",
        "A small cat sat on the mat. The cat was sleeping peacefully in the sun.",
        "Mom and Dad went to the big house. They were excited to see their friends.",
        "The girl played with her favorite toy. She smiled and laughed with joy.",
        "A brown dog jumped over the fence. The dog was fast and energetic."
    ]
    
    # Create mapper
    mapper = GibberishLanguageMapper(seed=42)
    
    print("=== GIBBERISH LANGUAGE MAPPER DEMO ===\n")
    
    # Test simple translation first
    simple_test = "The cat sat. The dog ran."
    print(f"Testing simple case:")
    print(f"Original:   '{simple_test}'")
    
    gibberish_simple = mapper.translate_text(simple_test, 'to_gibberish')
    print(f"Gibberish:  '{gibberish_simple}'")
    
    back_simple = mapper.translate_text(gibberish_simple, 'to_english')
    print(f"Back:       '{back_simple}'")
    print(f"Round-trip successful: {simple_test == back_simple}")
    print()
    
    # Show word mappings created
    print("Word mappings created:")
    for word, gibberish in mapper.english_to_gibberish.items():
        print(f"  {word} → {gibberish}")
    print()
    
    # Create training dataset
    dataset = mapper.create_training_dataset(sample_stories)
    
    print(f"Total vocabulary size: {len(mapper.english_to_gibberish)}")
    
    print("\n=== TRANSLATION EXAMPLES ===")
    test_sentence = "The happy boy ran quickly to his mother."
    gibberish_version = mapper.translate_text(test_sentence, 'to_gibberish')
    back_to_english = mapper.translate_text(gibberish_version, 'to_english')
    
    print(f"Original:   {test_sentence}")
    print(f"Gibberish:  {gibberish_version}")
    print(f"Back:       {back_to_english}")
    
    print(f"\nPerfect round-trip: {test_sentence == back_to_english}")
    
    print("\n=== PARALLEL DATASET SAMPLE ===")
    for i in range(min(2, len(dataset['english_stories']))):
        print(f"English:   {dataset['english_stories'][i]}")
        print(f"Gibberish: {dataset['gibberish_stories'][i]}")
        print()
    
    # Export mappings
    mapper.export_mappings('demo_mappings.json')
    
    return mapper, dataset

def test_simple_case():
    """Test the exact issue the user reported"""
    mapper = GibberishLanguageMapper(seed=42)
    
    test_text = "The cat sat. The dog ran."
    print(f"Testing: '{test_text}'")
    
    # Use debug translate to see what's happening
    gibberish = mapper.debug_translate(test_text)
    
    # Also test the regular translate
    print("\nUsing regular translate_text:")
    regular_result = mapper.translate_text(test_text, 'to_gibberish')
    print(f"Regular result: '{regular_result}'")
    
    return mapper

if __name__ == "__main__":
    print("=== SIMPLE TEST ===")
    test_simple_case()
    print("\n" + "="*50 + "\n")
    demonstrate_usage()

=== SIMPLE TEST ===
Testing: 'The cat sat. The dog ran.'
Debugging translation of: 'The cat sat. The dog ran.'
Words processed: The → gu, cat → dy, sat → feda, The → gu, dog → nu, ran → cam
Result: 'Gu dy feda. Gu nu cam.'

Using regular translate_text:
Regular result: 'Gu dy feda. Gu nu cam.'


=== GIBBERISH LANGUAGE MAPPER DEMO ===

Testing simple case:
Original:   'The cat sat. The dog ran.'
Gibberish:  'Gu dy feda. Gu nu cam.'
Back:       'The cat sat. The dog ran.'
Round-trip successful: True

Word mappings created:
  the → gu
  cat → dy
  sat → feda
  dog → nu
  ran → cam

Total vocabulary size: 52

=== TRANSLATION EXAMPLES ===
Original:   The happy boy ran quickly to his mother.
Gibberish:  Gu ilnu if cam wrywekwota ic myk pubpamu.
Back:       The happy boy ran quickly to his mother.

Perfect round-trip: True

=== PARALLEL DATASET SAMPLE ===
English:   The little boy ran to the park. He was very happy and played with his red ball.
Gibberish: Gu iply if cam ic gu unxiw. Wub cizo 

In [20]:
mapper = GibberishLanguageMapper(seed=42)

Step 2: Tokenize the Dataset

In this step, we will do the following:

(1) Tokenize the dataset into tokenIDs.

(2) Create a file called "train.bin" and "validtion.bin" where we will store the tokenIDs from the entire dataset.

(3) We make sure the tokenIDs are stored on a disk, rather than on the RAM for efficient computations.

In [14]:
!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

# Some functions from https://github.com/karpathy/nanoGPT/blob/master/data/openwebtext/prepare.py

#def process(example):
#    ids = enc.encode_ordinary(example['text']) # encode_ordinary ignores any special tokens
#    out = {'ids': ids, 'len': len(ids)}
#    return out
def process(example):
    text = example.get('text') # Safeley get the 'text' value
    if text is not None:

        # Translate text to Gibberish
        gibberish_text = mapper.translate_text(text, direction='to_gibberish')
        # ids = enc.encode_ordinary(text)  # encode_ordinary ignores any special tokens
        ids = enc.encode_ordinary(gibberish_text)  # encode_ordinary ignores any special tokens
        out = {'ids': ids, 'len': len(ids)}
        return out
    else:
        # Handle the case where 'text' is None (e.g., skip the sample, log a warning)
        print("Warning: Skipping sample with None 'text' value.")  # You can customize this
        return {'ids': [], 'len': 0} # Return empty if skipping

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the splits",
        num_proc=8,
        )
    # concatenate all the ids in each dataset into one large file we can use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16 # (can do since enc.max_token_value == 50256 is < 2**16)
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # Batch together samples for faster write
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

Step 3: Create Input-Output batches for the dataset

In [ ]:
# Some functions from https://github.com/karpathy/nanoGPT/blob/master/train.py with slight modifications
def get_batch(split):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('validation.bin', dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

Step 4: Define the SLM Model Architecture

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [9]:
config = GPTConfig(
    vocab_size=50257,     # use the tokenizer's vocab size
    block_size=128,       # or whatever context size you're training with
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1,
    bias=True
)

model = GPT(config)

Step 5: Define the loss function

In [ ]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

Step 6: Define SLM Training Configuration Part 1

In [ ]:
# Training Config
import torch
from contextlib import nullcontext

learning_rate = 1e-4 #more stable training, earlier 1e-4
max_iters = 20000 #increase from 25000
warmup_steps = 1000 #smoother initial train, earlier 100
min_lr = 5e-4 #lower rate, earlier 5e-4
eval_iters = 500 # increased from 100
batch_size = 32 # changed from 16, better gradient estimate
block_size = 128 #changed from 64, capture longer range dependencies

gradient_accumulation_steps = 32 # reduced from 50

device =  "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
# note: float16 data type will automatically use a GradScaler

# How to use autocast https://wandb.ai/wandb_fc/tips/reports/How-To-Use-Autocast-in-PyTorch--VmlldzoyMTk4NTky
#dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

Step 7: Define SLM Training Configuration Part 2

In [ ]:
from torch.optim.lr_scheduler import LinearLR,SequentialLR, CosineAnnealingLR

##PUT IN WEIGHT DECAY, CHANGED BETA2 to 0.95
optimizer =  torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9) #weight decay for regularization

scheduler_warmup = LinearLR(optimizer, total_iters = warmup_steps) #Implement linear warmup
scheduler_decay = CosineAnnealingLR(optimizer,T_max = max_iters - warmup_steps, eta_min = min_lr) #Implement lr decay
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps]) #Switching from warmup to decay

# https://stackoverflow.com/questions/72534859/is-gradscaler-necessary-with-mixed-precision-training-with-pytorch
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

Step 8: Pre-train the SLM

In [ ]:
best_val_loss = float('inf')
best_model_params_path = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []

# Ensure model is on the correct device
model = model.to(device)

# In your training loop
for epoch in tqdm(range(max_iters)):
    if epoch % eval_iters == 0 and epoch != 0:
        # Ensure estimate_loss uses the correct device
        losses = estimate_loss(model)
        print(f"Epoch {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"The current learning rate: {optimizer.param_groups[0]['lr']:.5f}")
        train_loss_list += [losses['train']]
        validation_loss_list += [losses['val']]

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    # Ensure X and y are on the correct device
    X, y = get_batch("train")
    X, y = X.to(device), y.to(device)

    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    scheduler.step()

Step 9: Plot the SLM Loss Function

In [ ]:
import matplotlib.pyplot as plt
train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]
validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]

plt.plot(train_loss_list_converted, 'g', label='train_loss')
plt.plot(validation_loss_list_converted, 'r', label='validation_loss')
plt.xlabel("Steps - Every 100 epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


Step 10: Run SLM Inference on our trained model

In [10]:
#Load the model
model = GPT(config)  # re-create the model with same config
device =  "cuda" if torch.cuda.is_available() else "cpu"
best_model_params_path = "best_model_params.pt"
model.load_state_dict(torch.load(best_model_params_path, map_location=torch.device(device))) # load best model states

<All keys matched successfully>

In [23]:
sentence = "Once upon a time there was a pumpkin."
gibberish_sentence = mapper.translate_text(sentence, direction='to_gibberish')
print(f"Sentence = {sentence}\n")
print(f"Gibberish Sentence = {gibberish_sentence}\n")

#context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
context = (torch.tensor(enc.encode_ordinary(gibberish_sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
full_story = enc.decode(y.squeeze().tolist())
print(f"Full Story = {full_story}\n")
print(f"Full Story in English = {mapper.translate_text(full_story, direction='to_english')}")

Sentence = Once upon a time there was a pumpkin.

Gibberish Sentence = Uwkig kjykaj qo fehefcu helhove cizo qo lyugbip.

Full Story = Uwkig kjykaj qo fehefcu helhove cizo qo lyugbip. Wub rbiratrify ic xo roxryje. Wub kdekaq ic dy ratrify ly uwmigjo gu ly usnic'ucg xaumh ig mememhy.

Hbuhuc ir ly wuj hagehso, helhove, gu ixne cizo ir ly guubxik. Iply wpoweb kolkyvo fnufiffupy hbuhuc geunz ly ly rwurog wje insulting is urk baujq. Helhove, gu fqafin nuucdba ujlir ly ultitqa uxg ratrify gu zipezxyzal huk uxg.

Ufdin cizo urk ukjisja, ib jaukg ic ugwip hbuhuc gu wabewni uxfih, uxdihxo. W

Full Story in English = Once upon a time there was a pumpkin. Wub rbiratrify ic xo roxryje. Wub kdekaq ic dy ratrify ly uwmigjo gu ly usnic'ucg xaumh ig mememhy.

Hbuhuc ir ly wuj hagehso, there, gu ixne was ir ly guubxik. Iply wpoweb kolkyvo fnufiffupy hbuhuc geunz ly ly rwurog wje insulting is urk baujq. There, gu fqafin nuucdba ujlir ly ultitqa uxg ratrify gu zipezxyzal huk uxg.

Ufdin was urk ukjisja, 

In [ ]:
sentence = "A little girl went to the woods."
context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
print(enc.decode(y.squeeze().tolist()))